# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"FAIR2 Citation: {getattr(metadata, 'citeAs', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We list all available record sets (`@id`), their associated fields, and the columns in each. All entities are referenced by their `@id`.

This information helps guide extraction and selection in following steps.

In [ ]:
# List record sets and their fields
record_sets = dataset.record_sets

print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {getattr(rs, 'description', '-')}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.id} (name: {field.name}, dataType: {getattr(field, 'dataType', '-')})")
    print("  Columns:")
    for col in getattr(rs, 'columns', []):
        print(f"    - {col.id} (name: {getattr(col, 'name', '-')}, dataType: {getattr(col, 'dataType', '-')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Reference record sets and fields by their `@id` as listed above. We will extract the main record set containing the survey responses for analysis, and show the columns for exploratory work.

In [ ]:
# Extract data from all record sets
import pprint

# List of record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Number of records: {len(df)}\n")

# For demonstration, pick the first record set as the main one
main_record_set_id = record_set_ids[0] if record_set_ids else None

if main_record_set_id:
    df_main = dataframes[main_record_set_id]
    print(f"Previewing records from {main_record_set_id}:")
    display(df_main.head())
else:
    print("No record set found in the schema.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping.

Let's choose a numeric field from the main record set, filter records, normalize it, and group by a key attribute. All field names are referenced by their `@id`. 

In [ ]:
# Example EDA for a numeric field (PHQ-9 total score) grouped by gender

if main_record_set_id:
    # PHQ-9 total score field's @id as per the dataset schema (if available)
    numeric_field_id = None
    group_field_id = None

    # Try to automatically pick PHQ-9/GAD-7/PSQ/gender
    columns = dataframes[main_record_set_id].columns
    preferred_numeric_columns = [
        'cr:field:phq9_total', 'cr:field:phq_9_total_score', 'cr:field:gad7_total', 'cr:field:psq_total', 'phq9_total', 'phq_9_total', 'PSQ_TotalScore',
        'cr:field:PHQ9_TOTAL', 'cr:field:GAD7_TOTAL', 'cr:field:PSQ_SUM', 'cr:field:psq_sum', 'sum_phq9', 'sum_gad7', 'sum_psq'
    ]
    for col in preferred_numeric_columns:
        if col in columns:
            numeric_field_id = col
            break
    # Default to first float/int-like field if not found
    if numeric_field_id is None:
        for col in columns:
            if pd.api.types.is_numeric_dtype(dataframes[main_record_set_id][col]):
                numeric_field_id = col
                break

    # Try to pick group field (gender if available)
    for cand in ['cr:field:gender', 'gender', 'cr:field:sex', 'sex']:
        if cand in columns:
            group_field_id = cand
            break
    if numeric_field_id is None:
        print("No appropriate numeric field found for EDA.")
    else:
        df = dataframes[main_record_set_id]
        df = df.dropna(subset=[numeric_field_id])

        # Convert to numeric in case
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        df = df.dropna(subset=[numeric_field_id])

        threshold = df[numeric_field_id].mean() + df[numeric_field_id].std()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df[[numeric_field_id] + ([group_field_id] if group_field_id else [])].head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std()!=0 else 1)
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id} (after filtering):")
            display(grouped_df)
else:
    print("No main record set loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We plot the distribution of the selected numeric mental health score and visualize the group means by gender, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna(), bins=20, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
    
    if group_field_id and group_field_id in dataframes[main_record_set_id].columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(
            x=dataframes[main_record_set_id][group_field_id], 
            y=dataframes[main_record_set_id][numeric_field_id]
        )
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the [FAIR² Mental Health Survey Dataset from Kilifi County](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) with `mlcroissant`. We reviewed the schema programmatically, extracted records for analysis, filtered and normalized key survey scores, grouped by demographic attributes, and visualized distributions. All data elements were referenced by their Croissant `@id` identifiers.

Further analysis can be performed, such as longitudinal studies, predicting risk with ML, or drilling into other survey instruments or demographic subgroups. Follow best practices for privacy when working with personal health data.